# Chapter 12 - Code Organization at Small Scale

#### 1. Mechanical Refresher
A module is a Python object that holds names, and an import statement finds a module, executes it once if needed, stores it in `sys.modules`, and binds one or more names in the importing code. A package is a module that can contain submodules; in file-backed projects `__init__.py` marks and initializes a package, while this notebook simulates modules in memory so the import mechanics can be drilled without turning this chapter into file I/O practice.

#### 2. Minimal Working Example

In [ ]:
import sys
import types
from importlib import import_module

mini = types.ModuleType("mini")
mini.__path__ = []
metrics = types.ModuleType("mini.metrics")
def accuracy(preds, labels):
    return sum(int(p == y) for p, y in zip(preds, labels)) / len(labels)
metrics.accuracy = accuracy
mini.metrics = metrics
sys.modules["mini"] = mini
sys.modules["mini.metrics"] = metrics
loaded = import_module("mini.metrics")
assert loaded.accuracy([1, 0, 1], [1, 1, 1]) == 2 / 3

`types.ModuleType` creates module objects, `sys.modules` is the import cache, and `import_module` retrieves the registered module. In a file-backed project, the same names would come from files such as `mini/__init__.py` and `mini/metrics.py`.

#### 3. Modify Drills

**Modify Drill 1.** Add another function to a module and import the module object.

In [ ]:
import sys
import types
from importlib import import_module

pkg = types.ModuleType("org_demo_a")
pkg.__path__ = []
helpers = types.ModuleType("org_demo_a.helpers")
helpers.double = lambda x: x * 2
pkg.helpers = helpers
sys.modules["org_demo_a"] = pkg
sys.modules["org_demo_a.helpers"] = helpers

loaded = import_module("org_demo_a.helpers")
assert loaded.double(4) == 8
print("passed")

**Modify Drill 2.** Bind a selected imported name locally and predict which namespace changes.

In [ ]:
import sys
import types
from importlib import import_module

module = types.ModuleType("org_demo_b")
module.value = 10
sys.modules["org_demo_b"] = module

loaded = import_module("org_demo_b")
local_value = loaded.value
loaded.value = 20
assert local_value == 10
assert loaded.value == 20
print("passed")

**Modify Drill 3.** Simulate package-level exports like an `__init__.py`.

In [ ]:
import sys
import types
from importlib import import_module

package = types.ModuleType("org_demo_c")
package.__path__ = []
package.VERSION = "1.0"
sys.modules["org_demo_c"] = package

loaded = import_module("org_demo_c")
assert loaded.VERSION == "1.0"
print("passed")

#### 4. Break-and-Fix Drills

**Break-and-Fix Drill 1.** Break it by importing a name before registering it in `sys.modules`. Predict the module-not-found symptom, then register before importing.

In [ ]:
import sys
import types
from importlib import import_module

module = types.ModuleType("org_demo_d")
module.answer = 42
sys.modules["org_demo_d"] = module

loaded = import_module("org_demo_d")
assert loaded.answer == 42
print("passed")

**Break-and-Fix Drill 2.** Break it by forgetting to attach a submodule to its package. Predict why package attribute access fails, then attach it.

In [ ]:
import sys
import types
from importlib import import_module

package = types.ModuleType("org_demo_e")
package.__path__ = []
submodule = types.ModuleType("org_demo_e.sub")
submodule.name = "sub"
package.sub = submodule
sys.modules["org_demo_e"] = package
sys.modules["org_demo_e.sub"] = submodule

loaded = import_module("org_demo_e")
assert loaded.sub.name == "sub"
print("passed")

#### 5. Self-Verification
Use asserts to verify imported modules expose the expected names and that repeated imports return the cached module object. For import-order drills, predict which name is bound locally, which name lives on the module, and which object is cached in `sys.modules`.

#### 6. Standalone Exercises

**Exercise 1.** Register and import a simple module. Expected behavior: `square(3) == 9`.

In [ ]:
import sys
import types
from importlib import import_module

module = types.ModuleType("exercise_math")
module.square = lambda x: None  # TODO: replace with a real square function.
sys.modules["exercise_math"] = module
loaded = import_module("exercise_math")
assert loaded.square(3) == 9
print("passed")

**Exercise 2.** Simulate an `__init__.py` export. Expected behavior: package exposes `VERSION`.

In [ ]:
import sys
import types
from importlib import import_module

package = types.ModuleType("exercise_pkg")
package.__path__ = []
package.VERSION = None  # TODO: set to "0.1".
sys.modules["exercise_pkg"] = package
loaded = import_module("exercise_pkg")
assert loaded.VERSION == "0.1"
print("passed")

**Exercise 3.** Register a package and submodule. Expected behavior: `metric.mae(...) == 1.0`.

In [ ]:
import sys
import types
from importlib import import_module

pkg = types.ModuleType("exercise_metrics")
pkg.__path__ = []
metric = types.ModuleType("exercise_metrics.metric")
metric.mae = lambda preds, targets: None  # TODO: replace with real MAE.
pkg.metric = metric
sys.modules["exercise_metrics"] = pkg
sys.modules["exercise_metrics.metric"] = metric
loaded = import_module("exercise_metrics.metric")
assert loaded.mae([2.0, 4.0], [1.0, 5.0]) == 1.0
print("passed")

**Exercise 4.** Show import caching. Expected behavior: both imports are the same object.

In [ ]:
import sys
import types
from importlib import import_module

module = types.ModuleType("exercise_cache")
module.count = 1
sys.modules["exercise_cache"] = module
first = import_module("exercise_cache")
second = None  # TODO: import the same module again.
assert first is second
print("passed")

**Exercise 5.** Split names by responsibility. Expected behavior: data module and metric module both work.

In [ ]:
import sys, types
from importlib import import_module

pkg = types.ModuleType("exercise_app")
pkg.__path__ = []
data = types.ModuleType("exercise_app.data")
metrics = types.ModuleType("exercise_app.metrics")
data.sample = None  # TODO: set to [1, 2].
metrics.count = lambda values: None  # TODO: replace with a count function.
sys.modules["exercise_app"] = pkg
sys.modules["exercise_app.data"] = data
sys.modules["exercise_app.metrics"] = metrics
assert import_module("exercise_app.data").sample == [1, 2]
assert import_module("exercise_app.metrics").count([1, 2]) == 2
print("passed")

#### 7. Applied AI/ML Drill
**ML to Python mirror:** splitting data helpers, metrics, and training utilities into modules is ordinary namespace organization around related functions. **Python to ML mirror:** ML projects commonly separate `data`, `models`, `metrics`, and `train` modules so training code imports stable interfaces instead of depending on one large script.

**Applied Drill.** Simulate a tiny ML package with `data` and `metrics` modules. Expected behavior: average absolute error is `0.5`.

In [ ]:
import sys
import types
from importlib import import_module

pkg = types.ModuleType("tiny_ml")
pkg.__path__ = []
data = types.ModuleType("tiny_ml.data")
metrics = types.ModuleType("tiny_ml.metrics")
data.predictions = [2.0, 3.0]
data.targets = [1.5, 3.5]
metrics.mae = lambda preds, targets: None  # TODO: replace with real MAE.
pkg.data = data
pkg.metrics = metrics
sys.modules["tiny_ml"] = pkg
sys.modules["tiny_ml.data"] = data
sys.modules["tiny_ml.metrics"] = metrics

loaded_data = import_module("tiny_ml.data")
loaded_metrics = import_module("tiny_ml.metrics")
actual = loaded_metrics.mae(loaded_data.predictions, loaded_data.targets)
assert actual == 0.5
print("passed:", actual)

#### 8. Common Bugs
- Confusing module names with local variable names: the symptom is changing a local binding while the module still has the old value.
- Forgetting package initialization or exports: the symptom is a submodule import working but package-level access missing.
- Creating circular imports in real projects: the symptom is partially initialized modules and missing names.
- Putting every helper in one module: the symptom is broad imports and unclear ownership.

#### 9. Compounding Drill

Combine code organization with dataclasses and protocols: put a config record and model class into separate simulated modules and import both.

In [ ]:
import sys
import types
from dataclasses import dataclass
from importlib import import_module

pkg = types.ModuleType("organized_ml")
pkg.__path__ = []
config_mod = types.ModuleType("organized_ml.config")
model_mod = types.ModuleType("organized_ml.model")

@dataclass
class Config:
    factor: float

class Scaler:
    def __init__(self, config: Config):
        self.config = config
    def predict(self, x: float) -> float:
        return 0.0  # TODO

config_mod.Config = Config
model_mod.Scaler = Scaler
pkg.config = config_mod
pkg.model = model_mod
sys.modules["organized_ml"] = pkg
sys.modules["organized_ml.config"] = config_mod
sys.modules["organized_ml.model"] = model_mod
LoadedConfig = import_module("organized_ml.config").Config
LoadedScaler = import_module("organized_ml.model").Scaler
model = LoadedScaler(LoadedConfig(3.0))
assert model.predict(2.0) == 6.0
print("passed")

#### 10. Chapter Project
**Goal:** Structure a small simulated ML utility package with separate namespaces for data, metrics, models, and training.

**Requirements:** create one package module and at least three submodules; register them in `sys.modules`; expose a dataclass config; expose one model-like class with `predict`; expose one metric; write a training/evaluation helper that imports and uses those pieces.

**Stretch Goals:** add a package-level export like `VERSION`; add a protocol for the model interface; add an async batch fetcher module and import it from the training helper.

**Evaluation Checklist:** each module owns related names only; imports use the registered module names; repeated imports use cached objects; the final evaluation assert passes; no file I/O is required in this notebook simulation.

In [ ]:
import sys
import types
from dataclasses import dataclass
from importlib import import_module

package = types.ModuleType("project_ml")
package.__path__ = []
data = types.ModuleType("project_ml.data")
models = types.ModuleType("project_ml.models")
metrics = types.ModuleType("project_ml.metrics")
train = types.ModuleType("project_ml.train")

# TODO: fill modules, register them, import them, and assert final metric.
result = None
assert result == 0.5
print("project check passed")

#### 11. Solutions Appendix

--- SOLUTIONS: DO NOT PEEK UNTIL ATTEMPTED ---

In [ ]:
# Standalone Exercises 1-2
import sys
import types
from importlib import import_module

module = types.ModuleType("exercise_math")
module.square = lambda x: x * x
sys.modules["exercise_math"] = module
assert import_module("exercise_math").square(3) == 9

package = types.ModuleType("exercise_pkg")
package.__path__ = []
package.VERSION = "0.1"
sys.modules["exercise_pkg"] = package
assert import_module("exercise_pkg").VERSION == "0.1"
print("solutions passed")

In [ ]:
# Standalone Exercises 3-5
import sys
import types
from importlib import import_module

pkg = types.ModuleType("exercise_metrics")
pkg.__path__ = []
metric = types.ModuleType("exercise_metrics.metric")
def mae(preds, targets):
    return sum(abs(p - t) for p, t in zip(preds, targets)) / len(preds)
metric.mae = mae
pkg.metric = metric
sys.modules["exercise_metrics"] = pkg
sys.modules["exercise_metrics.metric"] = metric
assert import_module("exercise_metrics.metric").mae([2.0, 4.0], [1.0, 5.0]) == 1.0

module = types.ModuleType("exercise_cache")
module.count = 1
sys.modules["exercise_cache"] = module
first = import_module("exercise_cache")
second = import_module("exercise_cache")
assert first is second

app = types.ModuleType("exercise_app")
app.__path__ = []
data = types.ModuleType("exercise_app.data")
metrics = types.ModuleType("exercise_app.metrics")
data.sample = [1, 2]
metrics.count = len
app.data = data
app.metrics = metrics
sys.modules["exercise_app"] = app
sys.modules["exercise_app.data"] = data
sys.modules["exercise_app.metrics"] = metrics
assert import_module("exercise_app.data").sample == [1, 2]
assert import_module("exercise_app.metrics").count([1, 2]) == 2
print("solutions passed")

In [ ]:
# Applied AI/ML Drill and Compounding Drill
import sys
import types
from dataclasses import dataclass
from importlib import import_module

pkg = types.ModuleType("tiny_ml")
pkg.__path__ = []
data = types.ModuleType("tiny_ml.data")
metrics = types.ModuleType("tiny_ml.metrics")
data.predictions = [2.0, 3.0]
data.targets = [1.5, 3.5]
metrics.mae = lambda preds, targets: sum(abs(p - t) for p, t in zip(preds, targets)) / len(preds)
pkg.data = data
pkg.metrics = metrics
sys.modules["tiny_ml"] = pkg
sys.modules["tiny_ml.data"] = data
sys.modules["tiny_ml.metrics"] = metrics
assert import_module("tiny_ml.metrics").mae(import_module("tiny_ml.data").predictions, import_module("tiny_ml.data").targets) == 0.5

organized = types.ModuleType("organized_ml")
organized.__path__ = []
config_mod = types.ModuleType("organized_ml.config")
model_mod = types.ModuleType("organized_ml.model")

@dataclass
class Config:
    factor: float

class Scaler:
    def __init__(self, config: Config):
        self.config = config
    def predict(self, x: float) -> float:
        return x * self.config.factor

config_mod.Config = Config
model_mod.Scaler = Scaler
organized.config = config_mod
organized.model = model_mod
sys.modules["organized_ml"] = organized
sys.modules["organized_ml.config"] = config_mod
sys.modules["organized_ml.model"] = model_mod
LoadedConfig = import_module("organized_ml.config").Config
LoadedScaler = import_module("organized_ml.model").Scaler
assert LoadedScaler(LoadedConfig(3.0)).predict(2.0) == 6.0
print("solutions passed")

In [ ]:
# Chapter Project approach
import sys
import types
from dataclasses import dataclass
from importlib import import_module

package = types.ModuleType("project_ml")
package.__path__ = []
package.VERSION = "0.1"
data = types.ModuleType("project_ml.data")
models = types.ModuleType("project_ml.models")
metrics = types.ModuleType("project_ml.metrics")
train = types.ModuleType("project_ml.train")

@dataclass
class Config:
    weight: float
    bias: float

class Linear:
    def __init__(self, config: Config):
        self.config = config
    def predict(self, x: float) -> float:
        return self.config.weight * x + self.config.bias

def mae(preds, targets):
    return sum(abs(p - t) for p, t in zip(preds, targets)) / len(preds)

def evaluate(model, xs, targets):
    preds = [model.predict(x) for x in xs]
    return mae(preds, targets)

data.xs = [1.0, 2.0]
data.targets = [3.5, 5.5]
models.Config = Config
models.Linear = Linear
metrics.mae = mae
train.evaluate = evaluate
package.data = data
package.models = models
package.metrics = metrics
package.train = train
sys.modules["project_ml"] = package
sys.modules["project_ml.data"] = data
sys.modules["project_ml.models"] = models
sys.modules["project_ml.metrics"] = metrics
sys.modules["project_ml.train"] = train

Data = import_module("project_ml.data")
Models = import_module("project_ml.models")
Train = import_module("project_ml.train")
model = Models.Linear(Models.Config(weight=2.0, bias=1.0))
result = Train.evaluate(model, Data.xs, Data.targets)
assert result == 0.5
print("project solution passed")